# Rust — Weather Data Analysis

This notebook covers:
1. Loading `weather.json` with `serde_json`
2. Data inspection
3. Data cleaning (type coercion, missing values)
4. Descriptive statistics
5. Outlier detection (IQR & Z-score)
6. Data processing & filtering
7. Correlation analysis

> **Runtime:** This notebook requires the [evcxr](https://github.com/evcxr/evcxr) Rust kernel.  
> See the `.devcontainer/` folder to run this in GitHub Codespaces with zero local setup.

## Setup — Load Dependencies

The `:dep` directive tells evcxr to download and compile crates from crates.io.  
Run this cell **first** — it may take ~30 seconds on first run while Cargo compiles.

In [5]:
:dep serde = { version = "1", features = ["derive"] }
:dep serde_json = "1"

use serde_json::Value;
use std::fs;

println!("Dependencies loaded.");

Dependencies loaded.


## 1. Loading Weather Data

We read `weather.json` with `std::fs` and parse it using `serde_json`.  
Each record is kept as a `serde_json::Value` so we can handle the mixed types present in the raw data.

In [6]:
let raw = fs::read_to_string("weather.json").expect("File not found");
let records: Vec<Value> = serde_json::from_str(&raw).expect("JSON loading error");

println!("Records loaded : {}", records.len());

// Column names from first record (owned Strings to satisfy evcxr's 'static requirement)
let columns: Vec<String> = records[0].as_object().unwrap().keys().cloned().collect();
println!("Columns ({}): {:?}", columns.len(), columns);

Records loaded : 366
Columns (22): ["Cloud3pm", "Cloud9am", "Evaporation", "Humidity3pm", "Humidity9am", "MaxTemp", "MinTemp", "Pressure3pm", "Pressure9am", "RISK_MM", "RainToday", "RainTomorrow", "Rainfall", "Sunshine", "Temp3pm", "Temp9am", "WindDir3pm", "WindDir9am", "WindGustDir", "WindGustSpeed", "WindSpeed3pm", "WindSpeed9am"]


## 2. Data Inspection

Before cleaning, we print the first few records and identify which columns contain numeric data stored as strings.

In [7]:
// Print first 3 records
println!("--- First 3 records ---");
for r in records.iter().take(3) {
    println!("{}", serde_json::to_string_pretty(r).unwrap());
}

--- First 3 records ---
{
  "Cloud3pm": 7,
  "Cloud9am": 7,
  "Evaporation": 3.4,
  "Humidity3pm": 29,
  "Humidity9am": 68,
  "MaxTemp": 24.3,
  "MinTemp": 8,
  "Pressure3pm": 1015,
  "Pressure9am": 1019.7,
  "RISK_MM": 3.6,
  "RainToday": "No",
  "RainTomorrow": "Yes",
  "Rainfall": 0,
  "Sunshine": "6.3",
  "Temp3pm": 23.6,
  "Temp9am": 14.4,
  "WindDir3pm": "NW",
  "WindDir9am": "SW",
  "WindGustDir": "NW",
  "WindGustSpeed": "30",
  "WindSpeed3pm": 20,
  "WindSpeed9am": "6"
}
{
  "Cloud3pm": 3,
  "Cloud9am": 5,
  "Evaporation": 4.4,
  "Humidity3pm": 36,
  "Humidity9am": 80,
  "MaxTemp": 26.9,
  "MinTemp": 14,
  "Pressure3pm": 1008.4,
  "Pressure9am": 1012.4,
  "RISK_MM": 3.6,
  "RainToday": "Yes",
  "RainTomorrow": "Yes",
  "Rainfall": 3.6,
  "Sunshine": "9.7",
  "Temp3pm": 25.7,
  "Temp9am": 17.5,
  "WindDir3pm": "W",
  "WindDir9am": "E",
  "WindGustDir": "ENE",
  "WindGustSpeed": "39",
  "WindSpeed3pm": 17,
  "WindSpeed9am": "4"
}
{
  "Cloud3pm": 7,
  "Cloud9am": 8,
  "Evaporatio

()

In [9]:
// Detect columns that store numbers as JSON strings
let check_cols = [
    "MinTemp","MaxTemp","Rainfall","Evaporation","Sunshine",
    "WindGustSpeed","WindSpeed9am","WindSpeed3pm",
    "Humidity9am","Humidity3pm","Pressure9am","Pressure3pm",
    "Temp9am","Temp3pm","RISK_MM"
];

println!("{:<16} {:>10} {:>10}", "Column", "#Numeric", "#String");
println!("{}", "-".repeat(38));
for col in &check_cols {
    let (mut nums, mut strs) = (0usize, 0usize);
    for row in &records {
        match &row[col] {
            Value::Number(_) => nums += 1,
            Value::String(_) => strs += 1,
            _ => {}
        }
    }
    println!("{:<16} {:>10} {:>10}", col, nums, strs)
}

Column             #Numeric    #String
--------------------------------------
MinTemp                 366          0
MaxTemp                 366          0
Rainfall                366          0
Evaporation             366          0
Sunshine                  0        366
WindGustSpeed             0        366
WindSpeed9am              0        366
WindSpeed3pm            366          0
Humidity9am             366          0
Humidity3pm             366          0
Pressure9am             366          0
Pressure3pm             366          0
Temp9am                 366          0
Temp3pm                 366          0
RISK_MM                 366          0


()

## 3. Data Cleaning

### 3.1 Type Coercion
Several columns (`Sunshine`, `WindGustSpeed`, `WindSpeed9am`) store numbers as JSON strings.  
We define a helper `to_f64()` that handles both cases transparently.

### 3.2 Missing Values
We count `null` and unparseable values per column and decide how to handle them:  
- **Drop** rows with missing values in key modelling columns  
- **Impute** with the column mean for less critical columns

In [10]:
// Helper: extract f64 from a Value regardless of whether it's Number or String
fn to_f64(v: &Value) -> Option<f64> {
    match v {
        Value::Number(n) => n.as_f64(),
        Value::String(s) => s.trim().parse::<f64>().ok(),
        Value::Null => None,
        _ => None
    }
}

// Extract a full numeric column (None = missing / unparseable)
fn extract_col(records: &[Value], col: &str) -> Vec<Option<f64>> {
    records.iter().map(|r| to_f64(&r[col])).collect()
}

// Drop None values to get clean f64 vec
fn clean_col(records: &[Value], col: &str) -> Vec<f64> {
    extract_col(records, col).into_iter().flatten().collect()
}

println!("Helpers defined.");

Helpers defined.


In [12]:
// Missing value report after coercion
println!("{:<16} {:>8} {:>8} {:>10}", "Column", "Total", "Missing", "Missing%");
println!("{}", "-".repeat(44));
for col in &check_cols {
    let values = extract_col(&records, col);
    let missing = values.iter().filter(|v| v.is_none()).count();
    let percentage = missing as f64 / values.len() as f64 * 100.0;
    println!("{:<16} {:>8} {:>8} {:>10}", col, values.len(), missing, percentage);
}

Column              Total  Missing   Missing%
--------------------------------------------
MinTemp               366        0          0
MaxTemp               366        0          0
Rainfall              366        0          0
Evaporation           366        0          0
Sunshine              366        3 0.819672131147541
WindGustSpeed         366        2 0.546448087431694
WindSpeed9am          366        7 1.912568306010929
WindSpeed3pm          366        0          0
Humidity9am           366        0          0
Humidity3pm           366        0          0
Pressure9am           366        0          0
Pressure3pm           366        0          0
Temp9am               366        0          0
Temp3pm               366        0          0
RISK_MM               366        0          0


()

In [17]:
// Mean imputation helper
fn impute_mean(values: &[Option<f64>]) -> Vec<f64> {
    let clean: Vec<f64> = values.iter().flatten().copied().collect(); 
    let mean = clean.iter().sum::<f64>() / clean.len() as f64;
    values.iter().map(|v| v.unwrap_or(mean)).collect()
}

// Example: impute Evaporation
let evap_raw = extract_col(&records, "Evaporation");
let evap = impute_mean(&evap_raw);
println!("Evaporation — raw missing: {}", evap_raw.iter().filter(|v| v.is_none()).count());
println!("Evaporation — after imputation, len: {}, first 5: {:?}", evap.len(), &evap[..5]);

Evaporation — raw missing: 0
Evaporation — after imputation, len: 366, first 5: [3.4, 4.4, 5.8, 7.2, 5.6]


## 4. Descriptive Statistics

For each numeric column we compute: **n**, **mean**, **median**, **std dev**, **min**, **max**, **skewness**, and **kurtosis**.

In [ ]:
fn mean(v: &[f64]) -> f64 { v.iter().sum::<f64>() / v.len() as f64 }

fn median(v: &[f64]) -> f64 {
    let mut s = v.to_vec();
    s.sort_by(|a,b| a.partial_cmp(b).unwrap());
    len n = s.len();
    if n % 2 == 0 {
        (s[n / 2] + s[(n / 2) - 1]) / 2
    } else {
        s[n / 2]
    }
}

fn std_dev(v: &[f64]) -> f64 {
    let m = mean(v);
    (v.iter().map(|x| (x - m).powi(2)).sum::<f64>() / v.len() as f64).sqrt()
}

fn skewness(v: &[f64]) -> f64 {
    let m = mean(v);
    let std = std_dev(v);
    v.iter().map(|x| ((x - m) / std).powi(3)).sum::<f64>() / v.len() as f64
}

fn kurtosis(v: &[f64]) -> 

println!("Statistics functions defined.");

In [ ]:
let stat_cols = [
    "MinTemp","MaxTemp","Rainfall","Humidity9am",
    "Humidity3pm","Pressure9am","Temp9am","Temp3pm"
];

println!("{:<14} {:>6} {:>7} {:>7} {:>6} {:>7} {:>7} {:>7} {:>7}",
         "Column","n","mean","median","std","min","max","skew","kurt");
println!("{}", "-".repeat(72));
for col in &stat_cols {
    
}

## 5. Outlier Detection

### 5.1 IQR Method
A value is an outlier if it falls below $Q_1 - 1.5 \times IQR$ or above $Q_3 + 1.5 \times IQR$.

### 5.2 Z-score Method
A value is an outlier if $|z| > threshold$ (commonly 3.0).

In [ ]:
fn iqr_bounds(v: &[f64]) -> 


fn iqr_outliers(v: &[f64]) -> 

println!("{:<14} {:>10} {:>10} {:>10} {:>10}",
         "Column", "Lower", "Upper", "Outliers", "Outl%");
println!("{}", "-".repeat(56));
for col in &stat_cols {
    
}

In [ ]:
fn zscore_outliers(v: &[f64], threshold: f64) -> 

// Show Z-score outliers for Rainfall (often heavily skewed)
let rainfall = 
let outliers = 
println!("Rainfall — Z-score outliers (|z| > 3.0): {}", outliers.len());
println!("{:<8} {:>10} {:>10}", "Index", "Value", "Z-score");
println!("{}", "-".repeat(30));
for (i, val, z) in &outliers {
    println!("{:<8} {:>10.2} {:>10.3}", i, val, z);
}

In [ ]:
// Remove IQR outliers from MaxTemp and compare stats before/after
let max_temp = 
let (lo, hi) = 
let max_temp_clean: Vec<f64> = 

println!("MaxTemp — before outlier removal:");
println!("  n={}, mean={:.2}, std={:.2}, min={:.2}, max={:.2}",
    max_temp.len(), mean(&max_temp), std_dev(&max_temp),
    max_temp.iter().cloned().fold(f64::INFINITY, f64::min),
    max_temp.iter().cloned().fold(f64::NEG_INFINITY, f64::max));

println!("\nMaxTemp — after IQR outlier removal:");
println!("  n={}, mean={:.2}, std={:.2}, min={:.2}, max={:.2}",
    max_temp_clean.len(), mean(&max_temp_clean), std_dev(&max_temp_clean),
    max_temp_clean.iter().cloned().fold(f64::INFINITY, f64::min),
    max_temp_clean.iter().cloned().fold(f64::NEG_INFINITY, f64::max));

println!("\nRows removed: {}", max_temp.len() - max_temp_clean.len());

## 6. Data Processing & Filtering

### 6.1 Min-Max Normalization
Rescales values to $[0, 1]$: $x' = \frac{x - x_{min}}{x_{max} - x_{min}}$

### 6.2 Filtering
Select subsets of records by condition (e.g. rainy days, extreme heat).

In [ ]:
fn min_max_normalize(v: &[f64]) -> 

fn z_standardize(v: &[f64]) -> 

let max_temp = clean_col(&records, "MaxTemp");
let norm = min_max_normalize(&max_temp);
let std_vals = z_standardize(&max_temp);

println!("MaxTemp — original first 5: {:?}", &max_temp[..5]);
println!("MaxTemp — normalized  first 5: {:.3?}", &norm[..5]);
println!("MaxTemp — standardized first 5: {:.3?}", &std_vals[..5]);

In [ ]:
// Filter counts (avoid Vec<&Value> which borrows from records -- not 'static)
let n_rainy = 

let n_hot = 

let n_humid = 

let n = records.len() as f64;
println!("Total records     : {}", records.len());
println!("Rainy days        : {} ({:.1}%)", n_rainy, n_rainy as f64 / n * 100.0);
println!("Hot days (>35 C)  : {} ({:.1}%)", n_hot,   n_hot   as f64 / n * 100.0);
println!("High humidity >80%: {} ({:.1}%)", n_humid,  n_humid  as f64 / n * 100.0);

In [ ]:
// Compare MaxTemp stats: rainy vs non-rainy days
let rainy_max: Vec<f64> = 

let dry_max: Vec<f64> = 

println!("{:<12} {:>6} {:>8} {:>8} {:>8}",
         "RainToday", "n", "mean", "median", "std");
println!("{}", "-".repeat(44));
println!("{:<12} {:>6} {:>8.2} {:>8.2} {:>8.2}",
    "Yes", rainy_max.len(), mean(&rainy_max), median(&rainy_max), std_dev(&rainy_max));
println!("{:<12} {:>6} {:>8.2} {:>8.2} {:>8.2}",
    "No", dry_max.len(), mean(&dry_max), median(&dry_max), std_dev(&dry_max));

## 7. Correlation Analysis

We compute the **Pearson correlation coefficient** between pairs of numeric columns:

$$r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2 \cdot \sum (y_i - \bar{y})^2}}$$

In [ ]:
fn pearson(xs: &[f64], ys: &[f64]) -> 

println!("Pearson correlation function defined.");

In [ ]:
// Correlation matrix for key columns
// We align vectors pairwise (only rows where both values are present)
fn aligned_cols(records: &[Value], col_a: &str, col_b: &str) -> 

let corr_cols = ["MinTemp","MaxTemp","Humidity3pm","Rainfall","Pressure9am","Temp3pm"];

// Header
print!("{:<12}", "");
for c in &corr_cols { print!("{:>10}", &c[..c.len().min(9)]); }
println!();
println!("{}", "-".repeat(12 + 10 * corr_cols.len()));

// Matrix
for row in &corr_cols {
    print!("{:<12}", &row[..row.len().min(11)]);
    for col in &corr_cols {
        if row == col {
            print!("{:>10.3}", 1.0_f64);
        } else {
            let (a, b) = aligned_cols(&records, row, col);
            print!("{:>10.3}", pearson(&a, &b));
        }
    }
    println!();
}

In [ ]:
// Top correlated pairs (|r| > 0.5, excluding self-correlation)
let all_cols = ["MinTemp","MaxTemp","Rainfall","Evaporation",
                "Humidity9am","Humidity3pm","Pressure9am","Pressure3pm",
                "Temp9am","Temp3pm"];

let mut pairs: Vec<(&str, &str, f64)> = 
for i in 0..all_cols.len() {
    for j in (i+1)..all_cols.len() {
        
    }
}
pairs.sort_by(|a, b| b.2.abs().partial_cmp(&a.2.abs()).unwrap());

println!("Top correlated pairs (|r| > 0.5):");
println!("{:<14} {:<14} {:>8}", "Col A", "Col B", "r");
println!("{}", "-".repeat(38));
for (a, b, r) in &pairs {
    println!("{:<14} {:<14} {:>8.3}", a, b, r);
}